In [2]:
# Cell 0: Stream AJILE12 ECoG+pose data (Dandiset 000055) directly from DANDI via S3.
# Uses the ros3 HDF5 driver for remote streaming — no local download needed.
# Prints session metadata, available acquisition streams, and processing modules.
# NOTE: Will fail if ros3 driver is not compiled into your HDF5/h5py build,
#       or if the DANDI asset URL returns a 403 (authentication/access issue).
from dandi.dandiapi import DandiAPIClient
from pynwb import NWBHDF5IO

dandiset_id = "000055"
version = "0.220127.0436"

with DandiAPIClient() as client:
    # 1. Access the dataset version
    dandiset = client.get_dandiset(dandiset_id, version_id=version)
    
    # 2. Grab the first available NWB asset file as a sample
    asset = next(dandiset.get_assets())
    print(f"Streaming asset path: {asset.path}")
    
    # 3. Generate the S3 streaming URL
    s3_url = asset.get_content_url(follow_redirects=True, strip_query=True)

# 4. Stream the data directly from the S3 URL
with NWBHDF5IO(s3_url, mode="r", load_namespaces=True, driver="ros3") as io:
    nwbfile = io.read()
    
    # Print basic experimental metadata
    print(f"Session Description: {nwbfile.session_description}")
    print(f"Subject ID: {nwbfile.subject.subject_id}")
    
    # Inspect the available data fields
    print("\nAvailable Data Streams:")
    print(nwbfile.acquisition.keys())  # Usually contains raw/processed ECoG signals
    print(nwbfile.processing.keys())   # Usually contains pose/behavioral tracking data

ModuleNotFoundError: No module named 'dandi'

In [2]:
# Cell 1: Connect to DANDI, find the first NWB asset in Dandiset 000004 (Rutishauser lab —
# human single-unit + behavior), and download it locally for offline inspection.
# Then opens the local file and prints a full summary: metadata, acquisition shapes,
# processing modules, and trial intervals.
# WARNING: The NWB file is large (~several GB). The download line is commented out below.
#          To use this cell, manually download the file first and set local_nwb_path accordingly.

import os
from dandi.dandiapi import DandiAPIClient
from pynwb import NWBHDF5IO
import numpy as np

dandiset_id = "000004" # Changed to a different known-open dandiset for testing

with DandiAPIClient() as client:
    # 1. Access the dataset version (allowing the client to select the default latest version)
    dandiset = client.get_dandiset(dandiset_id)
    version = dandiset.version.identifier # Get the actual version used
    print(f"Using Dandiset {dandiset_id}, Version: {version}")

    # 2. Grab the first available NWB asset file as a sample
    #    We'll iterate to find an NWB file specifically
    asset = None
    for a in dandiset.get_assets():
        if a.path.endswith('.nwb'):
            asset = a
            break
    if asset is None:
        raise ValueError("No NWB asset found in the specified dandiset.")

    print(f"Selected asset path: {asset.path}")

    # 3. Generate the S3 streaming URL (not directly used for NWBHDF5IO with ros3 driver workaround)
    s3_url = asset.get_content_url(follow_redirects=True, strip_query=True)

    # 4. DOWNLOAD DISABLED — file is large. Set local_nwb_path manually if you have the file.
    local_nwb_path = os.path.basename(asset.path) # Use the original asset filename
    print(f"Would download: {asset.path} -> {local_nwb_path}")
    # asset.download(local_nwb_path)  # <-- COMMENTED OUT: triggers large file download

# 5. Open the locally downloaded NWB file — only runs if the file exists locally.
# Comment this entire block out if you skipped the download above.
with NWBHDF5IO(local_nwb_path, mode="r", load_namespaces=True) as io:
    nwbfile = io.read()

    print("\n--- NWB File Contents Summary ---")

    # --- Metadata ---
    print("\nMetadata:")
    print(f"  Session Description: {nwbfile.session_description}")
    print(f"  Subject ID: {nwbfile.subject.subject_id if nwbfile.subject else 'N/A'}")
    print(f"  Experimenter: {nwbfile.experimenter}")
    print(f"  Lab: {nwbfile.lab}")
    print(f"  Identifier: {nwbfile.identifier}")
    print(f"  Session Start Time: {nwbfile.session_start_time}")

    # --- Acquisition Data (e.g., ECoG, LFP) ---
    print("\nAcquisition Data (e.g., ECoG, LFP, raw signals):")
    if nwbfile.acquisition:
        for name, data_interface in nwbfile.acquisition.items():
            print(f"  - {name} ({type(data_interface).__name__})")
            if hasattr(data_interface, 'data') and data_interface.data is not None:
                data_shape = data_interface.data.shape
                print(f"    Shape: {data_shape}")
                # Print a small sample if it's not too large
                if data_shape and np.prod(data_shape) > 0:
                    try:
                        sample = data_interface.data[:min(data_shape[0], 5)] # Take first 5 rows or fewer
                        if len(data_shape) > 1: # If it's 2D or more, take a slice of columns too
                             sample = data_interface.data[:min(data_shape[0], 5), :min(data_shape[1], 5)]
                        print(f"    Sample data (first few elements):\n{sample}")
                    except Exception as e:
                        print(f"    Could not display sample data: {e}")
            if hasattr(data_interface, 'timestamps') and data_interface.timestamps is not None:
                print(f"    Timestamps shape: {data_interface.timestamps.shape}")
    else:
        print("  No acquisition data found.")

    # --- Processing Modules (e.g., behavioral data, epochs) ---
    print("\nProcessing Modules (e.g., behavioral, stimulus, epochs):")
    if nwbfile.processing:
        for module_name, module in nwbfile.processing.items():
            print(f"  - Module: {module_name} ({module.description})")
            if module.data_interfaces:
                for data_name, data_interface in module.data_interfaces.items():
                    print(f"    - {data_name} ({type(data_interface).__name__})")
                    if hasattr(data_interface, 'data') and data_interface.data is not None:
                         data_shape = data_interface.data.shape
                         print(f"      Shape: {data_shape}")
                         if data_shape and np.prod(data_shape) > 0:
                             try:
                                 sample = data_interface.data[:min(data_shape[0], 5)]
                                 if len(data_shape) > 1:
                                     sample = data_interface.data[:min(data_shape[0], 5), :min(data_shape[1], 5)]
                                 print(f"      Sample data (first few elements):\n{sample}")
                             except Exception as e:
                                 print(f"      Could not display sample data: {e}")
    else:
        print("  No processing modules found.")

    # --- Intervals (e.g., trials, epochs) ---
    print("\nIntervals (e.g., trials, epochs):")
    if nwbfile.intervals:
        for name, interval in nwbfile.intervals.items():
            print(f"  - {name} ({type(interval).__name__})")
            if hasattr(interval, 'start_time') and hasattr(interval, 'stop_time'):
                print(f"    Number of entries: {len(interval.start_time)}")
                print(f"    First entry: start={interval.start_time[0]}, stop={interval.stop_time[0]}")
    else:
        print("  No intervals found.")

    print("\n--- End of Summary ---")

Using Dandiset 000004, Version: 0.220126.1852
Selected asset path: sub-P10HMH/sub-P10HMH_ses-20060901_ecephys+image.nwb


Error 403 while sending HEAD request to https://api.dandiarchive.org/api/assets/38304fe9-5f37-4c0d-a741-9cf2bafab9ff/download/: 



--- NWB File Contents Summary ---

Metadata:
  Session Description: New/Old recognition task for ID: 7. 
  Subject ID: P10HMH
  Experimenter: None
  Lab: Rutishauser
  Identifier: H10_7
  Session Start Time: 2006-09-01 00:00:00-07:00

Acquisition Data (e.g., ECoG, LFP, raw signals):
  - events (AnnotationSeries)
    Shape: (1004,)
    Sample data (first few elements):
['55.0' '1.0' '2.0' '3.0' '20.0']
    Timestamps shape: (1004,)
  - experiment_ids (TimeSeries)
    Shape: (1004,)
    Sample data (first few elements):
[80. 80. 80. 80. 80.]
    Timestamps shape: (1004,)

Processing Modules (e.g., behavioral, stimulus, epochs):
  No processing modules found.

Intervals (e.g., trials, epochs):
  - trials (TimeIntervals)
    Number of entries: 200
    First entry: start=5850.331408, stop=5854.419808

--- End of Summary ---


In [3]:
# Cell 2: Quick structural inspection of the NWB file loaded in Cell 1.
# Prints top-level container keys: acquisition, processing modules, interval tables,
# subject demographics, stimulus keys, and whether a units (spike sorting) table exists.
print("Acquisition keys:", list(nwbfile.acquisition.keys()))
print("Processing modules:", list(nwbfile.processing.keys()))
print("Interval tables:", list(nwbfile.intervals.keys()))
print("Subject info:", nwbfile.subject)
print("Stimulus keys:", list(nwbfile.stimulus.keys()) if nwbfile.stimulus else [])
print("Units table:", hasattr(nwbfile, "units"), nwbfile.units if hasattr(nwbfile, "units") else None)

Acquisition keys: ['events', 'experiment_ids']
Processing modules: []
Interval tables: ['trials']
Subject info: subject pynwb.file.Subject at 0x4852813792
Fields:
  age: 37
  description: Left Lateral Frontal
  sex: M
  species: Human
  subject_id: P10HMH

Stimulus keys: ['StimulusPresentation']
Units table: True units pynwb.misc.Units at 0x4858187168
Fields:
  colnames: ['origClusterID' 'waveform_mean_encoding' 'waveform_mean_recognition'
 'IsolationDist' 'SNR' 'waveform_mean_sampling_rate' 'spike_times'
 'electrodes']
  columns: (
    origClusterID <class 'hdmf.common.table.VectorData'>,
    waveform_mean_encoding <class 'hdmf.common.table.VectorData'>,
    waveform_mean_recognition <class 'hdmf.common.table.VectorData'>,
    IsolationDist <class 'hdmf.common.table.VectorData'>,
    SNR <class 'hdmf.common.table.VectorData'>,
    waveform_mean_sampling_rate <class 'hdmf.common.table.VectorData'>,
    spike_times_index <class 'hdmf.common.table.VectorIndex'>,
    spike_times <class 'h

In [4]:
# Cell 3: Pull out the 'events' AnnotationSeries from the acquisition container.
# Verifies its type and prints the shape of both the event labels array and the
# timestamps array (1004 entries each in the sample file).
obj = nwbfile.acquisition["events"]
print(type(obj))
print("data shape:", obj.data.shape)
print("timestamps shape:", obj.timestamps.shape)

<class 'pynwb.misc.AnnotationSeries'>
data shape: (1004,)
timestamps shape: (1004,)


In [6]:
# Cell 4: Plots the event data from Cell 3 using matplotlib.
# Handles 1D, 2D, and higher-dimensional arrays automatically.
# NOTE: Will fail if the NWB file is already closed (HDF5 data is read lazily —
#       obj.data is only accessible while the 'with NWBHDF5IO(...)' context is open).
import matplotlib.pyplot as plt

# Plot obj.data
arr = np.asarray(obj.data)
plt.figure(figsize=(10, 4))
if arr.ndim == 1:
    plt.plot(arr)
    plt.title('obj.data (1D)')
    plt.xlabel('Sample index')
    plt.ylabel('Value')
elif arr.ndim == 2:
    if arr.shape[0] <= 10:
        for i in range(arr.shape[0]):
            plt.plot(arr[i, :], label=f'row {i}')
        plt.legend(fontsize='small', loc='best')
        plt.title(f'obj.data (2D, {arr.shape[0]} rows)')
    else:
        for i in range(min(10, arr.shape[0])):
            plt.plot(arr[i, :], label=f'row {i}')
        plt.legend(fontsize='small', loc='best')
        plt.title(f'obj.data (2D, first 10 of {arr.shape[0]} rows)')
    plt.xlabel('Sample index')
    plt.ylabel('Value')
else:
    plt.plot(arr.ravel())
    plt.title(f'obj.data ({arr.ndim}D flattened)')
    plt.xlabel('Flattened index')
    plt.ylabel('Value')

plt.tight_layout()
plt.show()

Matplotlib is building the font cache; this may take a moment.


RuntimeError: Unable to synchronously get dataspace (identifier is not of specified type)

In [7]:
# Cell 7: Utility function that systematically walks all major NWB containers
# (acquisition, processing, intervals, stimulus, units, devices, electrodes, trials)
# and prints their type, key count, and child keys. Useful as a first-pass map
# of any unknown NWB file before accessing specific data fields.

# Inspect the NWB structure: keys and types for major containers

def inspect_nwb_structure(nwbfile):
    containers = {
        'acquisition': nwbfile.acquisition,
        'processing': nwbfile.processing,
        'intervals': nwbfile.intervals,
        'stimulus': getattr(nwbfile, 'stimulus', None),
        'units': getattr(nwbfile, 'units', None),
        'devices': getattr(nwbfile, 'devices', None),
        'electrodes': getattr(nwbfile, 'electrodes', None),
        'trials': getattr(nwbfile, 'trials', None),
    }

    for name, container in containers.items():
        if container is None:
            print(f"{name}: None")
            continue
        try:
            keys = list(container.keys())
        except Exception:
            keys = None

        print(f"\n{name}: {type(container).__name__}")
        if keys is not None:
            print(f"  keys ({len(keys)}): {keys}")
        else:
            print("  (no keys available)")

        if name == 'acquisition' and keys:
            for key in keys:
                obj = container[key]
                print(f"    - {key}: {type(obj).__name__}")
        elif name == 'processing' and keys:
            for key in keys:
                module = container[key]
                print(f"    - module {key}: {type(module).__name__}")
                if hasattr(module, 'data_interfaces'):
                    print(f"      data_interfaces: {list(module.data_interfaces.keys())}")

inspect_nwb_structure(nwbfile)



acquisition: LabelledDict
  keys (2): ['events', 'experiment_ids']
    - events: AnnotationSeries
    - experiment_ids: TimeSeries

processing: LabelledDict
  keys (0): []

intervals: LabelledDict
  keys (1): ['trials']

stimulus: LabelledDict
  keys (1): ['StimulusPresentation']

units: Units
  (no keys available)

devices: LabelledDict
  keys (1): ['Neuralynx-cheetah']

electrodes: ElectrodesTable
  (no keys available)

trials: TimeIntervals
  (no keys available)


In [11]:
# Cell 8: Accesses specific data fields from the already-loaded NWB file using
# the known container keys discovered in Cells 2 and 7.
# Prints session description, per-key acquisition data shapes (e.g. ECoG 43.2M samples
# across 94 channels), available processing modules (e.g. 'behavior'), and
# interval table row counts (e.g. 1427 epochs, 156 reaching events).

# Access NWB data using actual keys from nwbfile (already loaded from cell 2)

print("=== Accessing Data from Already-Loaded NWB File ===\n")

# Session info
print(f"Session Description: {nwbfile.session_description}")
print(f"Subject ID: {nwbfile.subject.subject_id}")

# Acquisition data - print shape for each data key
print("\n--- Acquisition Data Shapes ---")
acq_keys = list(nwbfile.acquisition.keys())
print(f"Available keys: {acq_keys}\n")
for key in acq_keys:
    data_obj = nwbfile.acquisition[key]
    print(f"Key: '{key}'")
    print(f"  Type: {type(data_obj).__name__}")
    if hasattr(data_obj, 'data') and data_obj.data is not None:
        shape = data_obj.data.shape
        print(f"  Data shape: {shape}")
    if hasattr(data_obj, 'timestamps') and data_obj.timestamps is not None:
        ts_shape = data_obj.timestamps.shape
        print(f"  Timestamps shape: {ts_shape}")
    print()

# Processing modules
print("--- Processing Modules ---")
proc_keys = list(nwbfile.processing.keys())
print(f"Available modules: {proc_keys}")
if proc_keys:
    for module_name in proc_keys:
        module = nwbfile.processing[module_name]
        print(f"  Module '{module_name}': {module.description}")

# Intervals (trials, epochs)
print("\n--- Interval Tables ---")
interval_keys = list(nwbfile.intervals.keys())
print(f"Available intervals: {interval_keys}")
if interval_keys:
    for interval_name in interval_keys:
        interval = nwbfile.intervals[interval_name]
        print(f"  '{interval_name}': {len(interval)} rows")

=== Accessing Data from Already-Loaded NWB File ===

Session Description: no description
Subject ID: 01

--- Acquisition Data Shapes ---
Available keys: ['ECGL', 'ECGR', 'EOGL', 'EOGR', 'ElectricalSeries']

Key: 'ECGL'
  Type: TimeSeries
  Data shape: (43200000,)

Key: 'ECGR'
  Type: TimeSeries
  Data shape: (43200000,)

Key: 'EOGL'
  Type: TimeSeries
  Data shape: (43200000,)

Key: 'EOGR'
  Type: TimeSeries
  Data shape: (43200000,)

Key: 'ElectricalSeries'
  Type: ElectricalSeries
  Data shape: (43200000, 94)

--- Processing Modules ---
Available modules: ['behavior']
  Module 'behavior': pose data

--- Interval Tables ---
Available intervals: ['epochs', 'reaches']
  'epochs': 1427 rows
  'reaches': 156 rows


# AJILE12 Data Exploration (from BruntonUWBio/ajile12-nwb-data)

The repository https://github.com/BruntonUWBio/ajile12-nwb-data provides tools for analyzing the AJILE12 dataset (Dandiset 000055). 

Key resources:
- **Dashboards**: `stream_dashboard.ipynb` (streaming) and `offline_dashboard.ipynb` (offline)
- **Example analysis**: `Example_AJILE12_exploratory_analysis.ipynb` (neural-behavior correlation)
- **Setup**: Install via `pip install git+https://github.com/catalystneuro/brunton-lab-to-nwb.git`

In [13]:
# Cell 11: Neural-behavior correlation analysis for the AJILE12 dataset.
# Filters the coarse behavioral epoch table by a target activity (e.g. 'Eat'),
# then sets up parameters (subject, session, ECoG channel, keypoint, frequency band)
# for extracting high-gamma power and correlating it with wrist pose data.
# Based on the Example_AJILE12_exploratory_analysis.ipynb in ajile12-nwb-data.
# NOTE: Requires the AJILE12 NWB file (Dandiset 000055) to be loaded as nwbfile.

# Example: Extract neural-behavior correlation
# Based on Example_AJILE12_exploratory_analysis.ipynb

from scipy.signal import butter, sosfiltfilt, hilbert

# Parameters (adjust for different participants, behaviors, frequency bands)
sbj = 1
session = 3
behavior_type = 'Eat'  # or 'Walk', 'Sleep', etc.
neural_freq_range = [80, 100]  # Gamma band (Hz)
ecog_ch_num = 7  # Electrode over motor cortex
keypoint_of_interest = 'R_Wrist'

# Extract behavior epochs
coarse_labels = nwbfile.intervals['epochs'].to_dataframe()
coarse_labels = coarse_labels[coarse_labels['labels'].str.contains(behavior_type)]
print(f"Found {len(coarse_labels)} epochs of {behavior_type}")
print(coarse_labels.head())

# Example: Get acquisition data
print("\n--- Acquisition Data Available ---")
for key in nwbfile.acquisition.keys():
    obj = nwbfile.acquisition[key]
    print(f"{key}: {type(obj).__name__}")
    if hasattr(obj, 'data'):
        print(f"  Shape: {obj.data.shape}")

# Example: Get processing modules (behavioral data)
print("\n--- Processing Modules ---")
for module_name, module in nwbfile.processing.items():
    print(f"{module_name}: {module.description}")
    if hasattr(module, 'data_interfaces'):
        for data_name in module.data_interfaces.keys():
            print(f"  - {data_name}")

ModuleNotFoundError: No module named 'scipy'

In [ ]:
# Cell 12: Three-part visualization for the loaded NWB file.
#   1. Lists available pose keypoints from the 'behavior' processing module (e.g. R_Wrist).
#   2. Loads electrode metadata into a DataFrame and renders a 3D scatter plot
#      of electrode positions (x, y, z) on the cortical surface.
#   3. Iterates all interval tables (trials/epochs/reaches), printing their shape
#      and column names, plus the first row as a sample.
# Each block has its own try/except so failures in one don't block the others.

# Visualization: Plot behavioral data and electrode locations

# Plot 1: Show available keypoints (pose tracking)
try:
    behavior_module = nwbfile.processing['behavior']
    position = behavior_module.data_interfaces['Position']
    keypoints = list(position.spatial_series.keys())
    print(f"Available keypoints: {keypoints}")
except Exception as e:
    print(f"Could not access behavior data: {e}")

# Plot 2: Plot electrode information
try:
    electrodes_df = nwbfile.electrodes.to_dataframe()
    print(f"\nElectrode information available:")
    print(electrodes_df.head())
    print(f"\nTotal electrodes: {len(electrodes_df)}")
    
    # Show 3D electrode positions
    if all(col in electrodes_df.columns for col in ['x', 'y', 'z']):
        fig = plt.figure(figsize=(8, 6))
        ax = fig.add_subplot(111, projection='3d')
        ax.scatter(electrodes_df['x'], electrodes_df['y'], electrodes_df['z'], alpha=0.6)
        ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
        ax.set_title('Electrode Positions')
        plt.show()
except Exception as e:
    print(f"Could not visualize electrodes: {e}")
    
# Plot 3: Show intervals (trials, epochs)
print(f"\nInterval tables: {list(nwbfile.intervals.keys())}")
for interval_name in nwbfile.intervals.keys():
    interval_df = nwbfile.intervals[interval_name].to_dataframe()
    print(f"\n{interval_name}:")
    print(f"  Shape: {interval_df.shape}")
    print(f"  Columns: {list(interval_df.columns)}")
    if len(interval_df) > 0:
        print(f"  First row:\n{interval_df.iloc[0]}")